In [2]:
"""
Experiment 2: Forward Propagation and Backpropagation in Deep Learning

This single file contains:

Part A: From-scratch implementation of a 3-layer neural network
        (input -> hidden1 -> hidden2 -> output) using the same notation
        as the handwritten formulas: z^(l), h^(l), E^(l), W^(l), b^(l), L, S, J.

Part B: Simple evaluation & plotting for the from-scratch model.

Part C: PyTorch implementation of a Multi-Layer Perceptron for Fashion-MNIST
        with comments explaining forward/backward propagation logic.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import time

# ================================================================
# Part A: From-scratch 3-layer network (notation 100% like notes)
# ================================================================

# ----- 0. Global settings -----
torch.manual_seed(42)
device = torch.device("cpu")  # CPU is enough for from-scratch example

# ----- 1. Construct a toy dataset (for requirement 2.1) -----
# Here we create a simple regression dataset with structure:
#   y = x1 + 2*x2 - 3*x3 + 0.5*x4 + noise
# so that the network has something meaningful to learn.
N = 256           # number of training samples
input_dim = 4     # input X = (x1, x2, x3, x4)
output_dim = 1    # scalar output y

# X: shape (N, 4)
X = torch.randn(N, input_dim, device=device)

# True underlying linear function + noise
true_w = torch.tensor([[1.0, 2.0, -3.0, 0.5]], device=device)  # shape (1,4)
true_b = torch.tensor([[0.3]], device=device)                  # shape (1,1)

# y: shape (N, 1)
y = (X @ true_w.t()) + true_b + 0.1 * torch.randn(N, 1, device=device)

# ----- 2. Set layer sizes & initialize parameters (Requirement 2.2) -----
hidden1_dim = 8
hidden2_dim = 6

learning_rate = 0.05   # η
lambda_l2 = 1e-4       # λ
num_epochs = 200

# W^(1): weights from input layer to hidden layer 1, shape (hidden1_dim, 4)
W1 = torch.randn(hidden1_dim, input_dim, device=device) * 0.01
W1.requires_grad = False

# b^(1): bias for hidden layer 1, shape (hidden1_dim, 1)
b1 = torch.zeros(hidden1_dim, 1, device=device)
b1.requires_grad = False

# W^(2): weights from hidden layer 1 to hidden layer 2, shape (hidden2_dim, hidden1_dim)
W2 = torch.randn(hidden2_dim, hidden1_dim, device=device) * 0.01
W2.requires_grad = False

# b^(2): bias for hidden layer 2, shape (hidden2_dim, 1)
b2 = torch.zeros(hidden2_dim, 1, device=device)
b2.requires_grad = False

# W^(3): weights from hidden layer 2 to output layer, shape (output_dim, hidden2_dim)
W3 = torch.randn(output_dim, hidden2_dim, device=device) * 0.01
W3.requires_grad = False

# b^(3): bias for output layer, shape (output_dim, 1)
b3 = torch.zeros(output_dim, 1, device=device)
b3.requires_grad = False


# ----- 3. Activation function φ and derivative φ' (Requirement 2.3) -----
def phi(z):
    """
    Activation function φ(z) used for both hidden layers.
    Here φ is ReLU: φ(z) = max(0, z) applied element-wise.

    Input:
        z: pre-activation tensor z^(l) of shape (layer_size, N)
    Output:
        h: activation tensor h^(l) with the same shape
    """
    return torch.clamp(z, min=0.0)


def phi_prime(z):
    """
    Derivative φ'(z) for backpropagation.

    For ReLU:
        φ'(z) = 1 if z > 0
              = 0 if z <= 0

    Input:
        z: pre-activation tensor z^(l)
    Output:
        dphi: tensor of same shape with φ'(z)
    """
    return (z > 0).float()


# ----- 4. Loss function L and objective J (Requirement 2.4 & regularization) -----
def mse_loss(o, y_true):
    """
    Mean squared error loss:
        L = 1/2 * ||o - y||^2 averaged over all samples.

    Input:
        o      : model output (N, 1)
        y_true : ground-truth target (N, 1)
    Output:
        scalar value L
    """
    diff = o - y_true
    return 0.5 * (diff ** 2).mean()


# Lists to store losses for plotting
L_history = []
J_history = []

# ----- 5. Training loop: manual forward + backprop + SGD (Requirement 2.5 & 2.6) -----
for epoch in range(num_epochs):

    # ----- 5.1 Forward propagation -----
    # X^T has shape (4, N); each column is one input x
    X_T = X.t()

    # z^(1) = W^(1) x + b^(1)
    # W1: (h1, 4), X_T: (4, N) -> z1: (h1, N)
    z1 = W1 @ X_T + b1

    # h^(1) = φ(z^(1))
    h1 = phi(z1)  # shape (h1, N)

    # z^(2) = W^(2) h^(1) + b^(2)
    # W2: (h2, h1), h1: (h1, N) -> z2: (h2, N)
    z2 = W2 @ h1 + b2

    # h^(2) = φ(z^(2))
    h2 = phi(z2)  # shape (h2, N)

    # z^(3) = W^(3) h^(2) + b^(3)
    # W3: (1, h2), h2: (h2, N) -> z3: (1, N)
    z3 = W3 @ h2 + b3

    # o = z^(3) (identity activation at output for regression)
    # Transpose to shape (N, 1) to match y
    o = z3.t()

    # ----- 5.2 Compute loss L, regularization S, objective J -----
    # Data loss L(o, y)
    L = mse_loss(o, y)

    # Regularization term:
    #   S = (λ / 2) * ( ||W^(1)||_F^2 + ||W^(2)||_F^2 + ||W^(3)||_F^2 )
    S = (lambda_l2 / 2.0) * (
        torch.sum(W1 * W1) +
        torch.sum(W2 * W2) +
        torch.sum(W3 * W3)
    )

    # Total objective:
    #   J = L + S
    J = L + S

    L_history.append(L.item())
    J_history.append(J.item())

    # ----- 5.3 Backpropagation: compute E^(3), E^(2), E^(1) -----
    N_batch = X.shape[0]  # number of samples

    # E^(3): error at the output layer
    # For MSE and identity activation:
    #   ∂L/∂o = (o - y)
    #   ∂o/∂z^(3) = 1
    # => E^(3) = ∂J/∂z^(3) = (o - y)^T / N
    E3 = (o - y).t() / N_batch  # shape (1, N)

    # E^(2): error at second hidden layer
    #   E^(2) = (W^(3))^T E^(3) ⊙ φ'(z^(2))
    E2_linear = W3.t() @ E3            # (h2, N)
    E2 = E2_linear * phi_prime(z2)     # element-wise φ'(z^(2))

    # E^(1): error at first hidden layer
    #   E^(1) = (W^(2))^T E^(2) ⊙ φ'(z^(1))
    E1_linear = W2.t() @ E2            # (h1, N)
    E1 = E1_linear * phi_prime(z1)     # element-wise φ'(z^(1))

    # ----- 5.4 Gradients for W^(l), b^(l) -----
    # ∂J/∂W^(3) = E^(3) (h^(2))^T + λ W^(3)
    dW3 = E3 @ h2.t() + lambda_l2 * W3  # (1, h2)
    # ∂J/∂b^(3) = sum over samples of E^(3)
    db3 = E3.sum(dim=1, keepdim=True)   # (1, 1)

    # ∂J/∂W^(2) = E^(2) (h^(1))^T + λ W^(2)
    dW2 = E2 @ h1.t() + lambda_l2 * W2  # (h2, h1)
    # ∂J/∂b^(2) = sum over samples of E^(2)
    db2 = E2.sum(dim=1, keepdim=True)   # (h2, 1)

    # ∂J/∂W^(1) = E^(1) X^T + λ W^(1)
    dW1 = E1 @ X_T.t() + lambda_l2 * W1  # (h1, 4)
    # ∂J/∂b^(1) = sum over samples of E^(1)
    db1 = E1.sum(dim=1, keepdim=True)    # (h1, 1)

    # ----- 5.5 Gradient descent update -----
    # W^(l) ← W^(l) − η * ∂J/∂W^(l)
    W3 = W3 - learning_rate * dW3
    W2 = W2 - learning_rate * dW2
    W1 = W1 - learning_rate * dW1

    # b^(l) ← b^(l) − η * ∂J/∂b^(l)
    b3 = b3 - learning_rate * db3
    b2 = b2 - learning_rate * db2
    b1 = b1 - learning_rate * db1

    # ----- 5.6 Simple log -----
    if (epoch + 1) % 20 == 0:
        print(
            f"[From-scratch] Epoch {epoch + 1:3d} | "
            f"L = {L.item():.6f} | S = {S.item():.6f} | J = {J.item():.6f}"
        )

print("From-scratch training finished.")

# ================================================================
# Part B: Simple evaluation & plotting for from-scratch model
# (Requirement 2.7: evaluation / visualization — here we show loss curves)
# ================================================================

plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), L_history, label="Data loss L")
plt.plot(range(1, num_epochs + 1), J_history, label="Objective J = L + S")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("From-scratch MLP training loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("from_scratch_loss.png", dpi=150, bbox_inches="tight")
plt.close()
print("From-scratch loss curve saved to from_scratch_loss.png")

# ================================================================
# Part C: PyTorch MLP for Fashion-MNIST (Requirement 3)
# ================================================================

# ----- 1. Load Fashion-MNIST dataset (3.1) -----
def load_fashion_mnist(batch_size=128):
    """
    Construct training and test datasets for Fashion-MNIST.

    Input:
        batch_size: number of samples in each mini-batch
    Output:
        train_loader, test_loader: PyTorch DataLoader objects
    """
    transform = transforms.Compose([
        transforms.ToTensor(),                 # convert image to tensor
        transforms.Normalize((0.5,), (0.5,))   # normalize pixel values
    ])

    train_dataset = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.FashionMNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, test_loader


# ----- 2. Define the neural network with PyTorch (3.2) -----
class MLP(nn.Module):
    """
    Multi-Layer Perceptron for Fashion-MNIST.

    This implements the same forward-propagation idea as the formulas:
        x -> Linear -> ReLU -> Linear -> ReLU -> Linear -> ReLU -> Linear -> output

    Each nn.Linear corresponds to z^(l) = W^(l) a^(l-1) + b^(l),
    and each nn.ReLU corresponds to h^(l) = φ(z^(l)).
    """

    def __init__(self, input_size=784, hidden_sizes=[512, 256, 128], num_classes=10, dropout_rate=0.5):
        super(MLP, self).__init__()

        # Flatten layer: reshape 28x28 image into 784-dimensional vector
        # This corresponds to forming the input vector x = (x1, ..., x784).
        self.flatten = nn.Flatten()

        layers = []
        in_size = input_size

        # Build hidden layers
        for hidden_size in hidden_sizes:
            # nn.Linear implements z = W a + b
            layers.append(nn.Linear(in_size, hidden_size))
            # nn.ReLU implements activation h = φ(z)
            layers.append(nn.ReLU())
            # Dropout is a regularization technique to reduce overfitting
            layers.append(nn.Dropout(dropout_rate))
            in_size = hidden_size

        # Output layer: final Linear that maps to logits for 10 classes
        layers.append(nn.Linear(in_size, num_classes))

        # nn.Sequential stores the whole chain of layers so that
        # forward(x) simply applies them in order, just like composing
        # all z^(l) and h^(l) computations in the formulas.
        self.network = nn.Sequential(*layers)

        # Initialize weights analogous to "randomly initialize W and b" (3.3)
        self._initialize_weights()

    def _initialize_weights(self):
        """
        Initialize weights using Kaiming initialization for Linear layers.
        This affects W^(l) (weights) and b^(l) (biases) in all layers.
        """
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        """
        Forward propagation:

        Input:
            x: batch of images with shape (batch_size, 1, 28, 28)
        Output:
            logits: model output before softmax, shape (batch_size, 10)

        This corresponds conceptually to computing all z^(l) and h^(l)
        until the final layer.
        """
        x = self.flatten(x)   # reshape to (batch_size, 784), input vector x
        x = self.network(x)   # apply all Linear+ReLU+Dropout layers
        return x


# ----- 3. Train and test functions (3.4, 3.5, 3.6, 3.7) -----
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one epoch.

    For each mini-batch:
        1. Forward: compute output o = f(x; W, b)
        2. Compute loss L(o, y) (here CrossEntropyLoss)
        3. Backward: PyTorch computes gradients ∂L/∂W^(l), ∂L/∂b^(l)
           using backpropagation automatically (autograd).
        4. Optimizer step: update parameters like W^(l) <- W^(l) - η * gradient.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()           # reset accumulated gradients
        output = model(data)            # forward pass: compute logits
        loss = criterion(output, target)  # compute data loss L

        loss.backward()                 # backpropagate: compute gradients
        optimizer.step()                # parameter update using gradients

        running_loss += loss.item()
        # predicted class = argmax over logits
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    return running_loss / len(train_loader), 100.0 * correct / total


def evaluate(model, test_loader, criterion, device):
    """
    Evaluate the model on the test set.

    Uses the same forward pass as training but without gradient computation.
    """
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            test_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    return test_loss / len(test_loader), 100.0 * correct / total


def plot_results(train_losses, train_accs, test_losses, test_accs, model_name="MLP"):
    """
    Plot loss and accuracy curves for the PyTorch MLP model.
    This corresponds to the requirement to 'use graphical methods to plot accuracy'.
    """
    epochs = range(1, len(train_losses) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curve
    axes[0].plot(epochs, train_losses, "b-", label="Training Loss", linewidth=2)
    axes[0].plot(epochs, test_losses, "r-", label="Test Loss", linewidth=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{model_name} - Loss Curve")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy curve
    axes[1].plot(epochs, train_accs, "b-", label="Training Accuracy", linewidth=2)
    axes[1].plot(epochs, test_accs, "r-", label="Test Accuracy", linewidth=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title(f"{model_name} - Accuracy Curve")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{model_name.lower()}_results.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"{model_name} curves saved to {model_name.lower()}_results.png")


# ----- 4. Main function tying everything together -----
def main_pytorch():
    batch_size = 128
    learning_rate = 0.001
    num_epochs = 10   # keep small for quicker run

    print("\n[PyTorch MLP] Loading Fashion-MNIST dataset...")
    train_loader, test_loader = load_fashion_mnist(batch_size)
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}")

    # Create model
    model = MLP(
        input_size=784,
        hidden_sizes=[512, 256, 128],
        num_classes=10,
        dropout_rate=0.5
    ).to(device)

    print(f"\nModel Architecture:\n{model}")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    train_losses, train_accs = [], []
    test_losses, test_accs = [], []

    print("\n[PyTorch MLP] Starting training...")
    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        scheduler.step()

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_losses.append(test_loss)
        test_accs.append(test_acc)

        print(
            f"Epoch [{epoch:2d}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )

    total_time = time.time() - start_time
    print(f"\n[PyTorch MLP] Training completed in {total_time:.2f} seconds")
    print(f"Best Test Accuracy: {max(test_accs):.2f}%")

    # Plot curves
    plot_results(train_losses, train_accs, test_losses, test_accs, model_name="MLP")

    # Save model weights
    torch.save(model.state_dict(), "mlp_fashion_mnist.pth")
    print("Model saved to mlp_fashion_mnist.pth")


if __name__ == "__main__":
    # From-scratch part has already run above when the file was imported/executed.
    # Now run the PyTorch MLP experiment.
    main_pytorch()


[From-scratch] Epoch  20 | L = 6.200001 | S = 0.000000 | J = 6.200001
[From-scratch] Epoch  40 | L = 6.187378 | S = 0.000000 | J = 6.187378
[From-scratch] Epoch  60 | L = 6.185755 | S = 0.000000 | J = 6.185755
[From-scratch] Epoch  80 | L = 6.185545 | S = 0.000000 | J = 6.185545
[From-scratch] Epoch 100 | L = 6.185517 | S = 0.000000 | J = 6.185518
[From-scratch] Epoch 120 | L = 6.185512 | S = 0.000000 | J = 6.185512
[From-scratch] Epoch 140 | L = 6.185509 | S = 0.000000 | J = 6.185510
[From-scratch] Epoch 160 | L = 6.185507 | S = 0.000000 | J = 6.185507
[From-scratch] Epoch 180 | L = 6.185503 | S = 0.000000 | J = 6.185504
[From-scratch] Epoch 200 | L = 6.185500 | S = 0.000000 | J = 6.185501
From-scratch training finished.
From-scratch loss curve saved to from_scratch_loss.png

[PyTorch MLP] Loading Fashion-MNIST dataset...
Training samples: 60000
Test samples: 10000

Model Architecture:
MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_featu